## 准备数据

In [26]:
import os
import numpy as np
import tensorflow as tf
from tensorflow import keras
from tensorflow.keras import layers, optimizers, datasets

os.environ['TF_CPP_MIN_LOG_LEVEL'] = '2'  # or any {'0', '1', '2'}

def mnist_dataset():
    (x, y), (x_test, y_test) = datasets.mnist.load_data()
    #normalize
    x = x/255.0
    x_test = x_test/255.0
    
    return (x, y), (x_test, y_test)

In [27]:
print(list(zip([1, 2, 3, 4], ['a', 'b', 'c', 'd'])))

[(1, 'a'), (2, 'b'), (3, 'c'), (4, 'd')]


## 建立模型

In [28]:
class myModel:
    def __init__(self):
        ####################
        '''声明模型对应的参数'''
        ####################
        self.W1 = tf.Variable(shape=[28*28, 100], dtype=tf.float32,
                             initial_value=tf.random.uniform(shape=[28*28, 100], minval=-0.1, maxval=0.1))
        self.b1 = tf.Variable(shape=[100], dtype=tf.float32, initial_value=tf.zeros(shape=[100]))
        
        self.W2 = tf.Variable(shape=[100, 10], dtype=tf.float32,
                             initial_value=tf.random.uniform(shape=[100, 10], minval=-0.1, maxval=0.1))
        self.b2 = tf.Variable(shape=[10], dtype=tf.float32, initial_value=tf.zeros(shape=[10]))
        
    def __call__(self, x):
        ####################
        '''实现模型函数体，返回未归一化的logits'''
        ####################
        # 展平输入
        x = tf.reshape(x, [-1, 28*28])
        
        # 第一层
        h1 = tf.matmul(x, self.W1) + self.b1
        h1 = tf.nn.relu(h1)
        
        # 第二层
        logits = tf.matmul(h1, self.W2) + self.b2
        
        return logits
        
model = myModel()

optimizer = optimizers.Adam(learning_rate=0.001)

## 计算 loss

In [29]:
@tf.function
def compute_loss(logits, labels):
    return tf.reduce_mean(
        tf.nn.sparse_softmax_cross_entropy_with_logits(
            logits=logits, labels=labels))

@tf.function
def compute_accuracy(logits, labels):
    predictions = tf.argmax(logits, axis=1)
    return tf.reduce_mean(tf.cast(tf.equal(predictions, labels), tf.float32))

@tf.function
def train_one_step(model, optimizer, x, y):
    with tf.GradientTape() as tape:
        logits = model(x)
        loss = compute_loss(logits, y)

    # compute gradient
    trainable_vars = [model.W1, model.W2, model.b1, model.b2]
    grads = tape.gradient(loss, trainable_vars)
    optimizer.apply_gradients(zip(grads, trainable_vars))

    accuracy = compute_accuracy(logits, y)

    # loss and accuracy is scalar tensor
    return loss, accuracy

@tf.function
def test(model, x, y):
    logits = model(x)
    loss = compute_loss(logits, y)
    accuracy = compute_accuracy(logits, y)
    return loss, accuracy

## 实际训练

In [30]:
train_data, test_data = mnist_dataset()

batch_size = 128
train_dataset = tf.data.Dataset.from_tensor_slices((
    tf.cast(train_data[0], tf.float32), 
    tf.cast(train_data[1], tf.int64)
))
train_dataset = train_dataset.shuffle(60000).batch(batch_size)

for epoch in range(10):
    total_loss = 0.0
    total_accuracy = 0.0
    num_batches = 0
    
    for x_batch, y_batch in train_dataset:
        loss, accuracy = train_one_step(model, optimizer, x_batch, y_batch)
        total_loss += loss.numpy()
        total_accuracy += accuracy.numpy()
        num_batches += 1
    
    avg_loss = total_loss / num_batches
    avg_accuracy = total_accuracy / num_batches
    print('epoch', epoch, ': loss', avg_loss, '; accuracy', avg_accuracy)

loss, accuracy = test(model, 
                      tf.constant(test_data[0], dtype=tf.float32), 
                      tf.constant(test_data[1], dtype=tf.int64))

print('test loss', loss.numpy(), '; accuracy', accuracy.numpy())

2026-03-11 22:47:14.000009: W tensorflow/core/framework/local_rendezvous.cc:404] Local rendezvous is aborting with status: OUT_OF_RANGE: End of sequence


epoch 0 : loss 0.44579878694085934 ; accuracy 0.8825237650606932


2026-03-11 22:47:15.287411: W tensorflow/core/framework/local_rendezvous.cc:404] Local rendezvous is aborting with status: OUT_OF_RANGE: End of sequence


epoch 1 : loss 0.20303697679152113 ; accuracy 0.9424973347547975


2026-03-11 22:47:16.539293: W tensorflow/core/framework/local_rendezvous.cc:404] Local rendezvous is aborting with status: OUT_OF_RANGE: End of sequence


...

2026-03-11 22:47:22.747952: W tensorflow/core/framework/local_rendezvous.cc:404] Local rendezvous is aborting with status: OUT_OF_RANGE: End of sequence


epoch 7 : loss 0.06057540273297824 ; accuracy 0.9826759061833689


2026-03-11 22:47:23.920813: W tensorflow/core/framework/local_rendezvous.cc:404] Local rendezvous is aborting with status: OUT_OF_RANGE: End of sequence


epoch 8 : loss 0.05192848009222161 ; accuracy 0.9849191541864928
epoch 9 : loss 0.045643619140550526 ; accuracy 0.9871957178817374
test loss 0.08053403 ; accuracy 0.9753


2026-03-11 22:47:25.064939: W tensorflow/core/framework/local_rendezvous.cc:404] Local rendezvous is aborting with status: OUT_OF_RANGE: End of sequence
